# Project work part 3 - Data quality

- Author: Mehdi Jamali


## Links
- [GitHub repository](https://github.com/Mehdij1511/IND320-Mehdij1511)

- [Streamlit app](https://ind320mehdij1511.streamlit.app/)

- Video: Not recorded yet.

## Declaration of AI usage

I used the following AI tools while working on this project:
- **GitHub Copilot:**
    - Used for code completion, suggestions, and feedback during development.
    - Also used to help implement and test the Part 3 ERA5 downloader, data-quality analyses, Streamlit pages, and caching safeguards.
- **Gemini:**
    - Used for coding assistance and for finding relevant functions and methods.
- **NotebookLM (Gemini):**
    - Used as a research assistant to answer questions and explain topics related to the project.

I reviewed and adapted the suggestions, and tested the code before using it in the project.

**The AI tools were used within the guidelines of "K3 - Full use of AI" at NMBU.**

## Work log

I have kept this work on the temporary `part3` branch, separate from `main`. The supplied NVE shapefile has five regions (`RM`, `RN`, `RS`, `RV`, and `RØ`) in ETRS89 / UTM zone 33N. I load these regions with GeoPandas and take 1,000 reproducible sample points from each, using a fixed random seed and converting the coordinates to longitude and latitude. The mapping is NO1=RS+RØ, NO2=RS, NO3=RM, NO4=RM+RN, and NO5=RV. I checked the sample counts and confirmed that the points fall inside their source polygons. I have not yet confirmed a separate electricity-area boundary file.

The downloader requests hourly ERA5 precipitation and 2-metre temperature in UTC. It groups sampled coordinates into 0.25-degree grid cells and weights each cell by the number of sample points it represents. My teacher confirmed that Open-Meteo can handle up to 100 coordinates in one request, and the downloader can reuse data from the older, smaller cached batches. A live one-day test returned 24 hourly observations, and a five-location full-year test returned 8,784. Successful API responses are saved locally; a full area/year series is cached only after it passes the completeness checks. MongoDB sharing is supported, but I have not configured its connection here, so this run uses the local cache and SQLite usage ledger. Requests are spaced out to protect the API quota. If Open-Meteo returns HTTP 429, the downloader stops rather than retrying automatically; the server's quota response is more reliable than the local estimate.

At the last cache check, 155 of the 286 NO3 grid cells had full-year responses. Three cells had conflicting cached values, so I will not choose one arbitrarily. A mocked end-to-end test confirmed that 152 consistent cells can be reused and the other 134 requested in batches of 100 and 34, producing a complete weighted 8,784-hour series. This test did not make API requests. Open-Meteo has returned HTTP 429, so the complete weather series and its analyses are not available yet.

The Streamlit app has a Part 3 home page and two analysis pages. One explores STL and the spectrogram; the other shows precipitation SPC results and temperature LOF anomalies. Both let the user choose an electricity area, and a weather download starts only after confirmation. AppTest passed, including the confirmation and quota-warning flows, and the local app returned HTTP 200. The main unfinished part is the weather download: until the full NO3 series is available, I cannot run and check those analyses or export a complete PDF. I also still need to confirm both required NVE geography layers and record the screencast. I am leaving the incomplete weather results clearly identified rather than presenting them as finished.

This work is being developed on the temporary `part3` branch. Changes should remain separate from `main` until peer review and teacher feedback are complete.

## Load and check the NVE region shapefile

I keep the five region codes separate so each sampled location can be traced back to its source polygon. A fixed random seed makes the same sample reproducible.

## Inspect region names and sample locations

The helper reads the shapefile, checks the region codes, and creates the sample locations.

In [1]:
# Find the project root so imports work from either notebook location.
%matplotlib inline
from pathlib import Path
import sys
import pandas as pd

project_root = Path.cwd()
if not (project_root / "part3_data.py").is_file():
    project_root = project_root.parent
if not (project_root / "part3_data.py").is_file():
    raise FileNotFoundError("Could not find part3_data.py in the project.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Load the regions and take a reproducible sample from each one.
from part3_data import load_nve_regions, sample_region_locations

regions = load_nve_regions()
region_coordinates = sample_region_locations(
    regions,
    samples_per_region=1000,
    random_seed=42,
)

sample_counts = {code: len(points) for code, points in region_coordinates.items()}
assert set(sample_counts) == {"RM", "RN", "RS", "RV", "RØ"}
assert all(count == 1000 for count in sample_counts.values())

print(regions[["REGIONNAVN"]])
print("Sampled locations per region:", sample_counts)


               REGIONNAVN
REGION                   
RM      Region Midt-Norge
RN            Region Nord
RS             Region Sør
RV            Region Vest
RØ             Region Øst
Sampled locations per region: {'RM': 1000, 'RN': 1000, 'RS': 1000, 'RV': 1000, 'RØ': 1000}


## Download hourly ERA5 data for NO3

I map the electricity areas to the NVE regions as follows: NO1 uses RS and RØ, NO2 uses RS, NO3 uses RM, NO4 uses RM and RN, and NO5 uses RV. In other words, East is RS+RØ and North is RM+RN. For each region, the downloader starts with 1,000 sampled locations. If several samples fall in the same 0.25-degree ERA5 grid cell, it requests that cell once and weights its contribution by the number of samples there. Completed API batches and validated full area/year results are cached locally so later runs can reuse them.

To keep archive responses within the API's size limit, requests contain up to 100 grid cells and cover 14-day hourly intervals. Each validated batch is cached, and the complete area/year series is saved only after every requested timestamp is available.

In [ ]:
from part3_data import download_era5_data

# Request all hourly weather data for 2024; the downloader checks completeness.
weather_no3_2024 = download_era5_data(
    region_identifier="NO3",
    start_date="2024-01-01",
    end_date="2024-12-31",
    samples_per_region=1000,
    random_seed=42,
)
assert len(weather_no3_2024) == 8784
assert weather_no3_2024["time"].is_monotonic_increasing
assert weather_no3_2024["time"].is_unique
assert weather_no3_2024[["precipitation", "temperature_2m"]].notna().all().all()

# Save the complete series locally so future runs can reuse it.
weather_cache_file = project_root / "data" / "era5_cache" / "NO3_ERA5_2024.csv"
weather_cache_file.parent.mkdir(parents=True, exist_ok=True)
weather_no3_2024.to_csv(weather_cache_file, index=False)

print(f"Downloaded {len(weather_no3_2024):,} hourly observations.")
print(f"Cached complete area series at {weather_cache_file}.")
weather_no3_2024.head()


## Precipitation outliers and temperature anomalies

The precipitation analysis uses a high-pass DCT to calculate SATV and robust SPC limits, while the temperature analysis identifies anomalies with Local Outlier Factor. SATV is used to classify outliers but is not plotted.

In [ ]:
from IPython.display import display
from part3_analysis import plot_precipitation_outliers, plot_temperature_anomalies

# Use DCT-derived SATV to classify outliers; the plot shows precipitation and limits.
precipitation_figure, precipitation_summary, precipitation_outliers = (
    plot_precipitation_outliers(
        weather_no3_2024,
        frequency_cutoff=52,
        n_standard_deviations=3,
    )
)
assert precipitation_summary["observations"] == len(weather_no3_2024)
assert len(precipitation_outliers) == precipitation_summary["outliers"]
display(precipitation_figure)
display(pd.Series(precipitation_summary, name="Precipitation SPC summary"))
display(precipitation_outliers.head(10))

# Use Local Outlier Factor to flag unusual temperatures (1% expected by default).
temperature_figure, temperature_summary, temperature_anomalies = (
    plot_temperature_anomalies(weather_no3_2024, contamination=0.01)
)
assert temperature_summary["observations"] == len(weather_no3_2024)
assert len(temperature_anomalies) == temperature_summary["anomalies"]
display(temperature_figure)
display(pd.Series(temperature_summary, name="Temperature LOF summary"))
display(temperature_anomalies.head(10))


## Reservoir-filling STL and spectrogram

Weekly filling levels for electricity area 3 (NO3) are retrieved from Magasinstatistikk. STL and the spectrogram use the same time series.

In [ ]:
import matplotlib.pyplot as plt
from nve_data import load_reservoir_data
from part3_analysis import plot_spectrogram, plot_stl_decomposition

# Use the NO3 reservoir-filling series (area 3) for this analysis.
reservoir_data = load_reservoir_data()
no3_reservoir_data = reservoir_data.loc[
    (reservoir_data["omrType"] == "EL") & (reservoir_data["omrnr"] == 3)
].sort_values("dato_Id")
no3_filling = no3_reservoir_data.set_index("dato_Id")["fyllingsgrad"]

stl_figure, stl_result = plot_stl_decomposition(
    no3_filling,
    seasonal=13,
    trend=53,
    robust=True,
)
assert len(stl_result.trend) == len(no3_filling)
display(stl_figure)
plt.close(stl_figure)

spectrogram_figure = plot_spectrogram(
    no3_filling,
    window_length=64,
    window_overlap=32,
)
assert spectrogram_figure.axes
display(spectrogram_figure)
plt.close(spectrogram_figure)
